# PortfolioRiskDiagnosis

This notebook takes a first object-oriented stab at the `PortfolioRiskDiagnosis` layer.

The goal is not to produce final recommendations yet. The goal is to transform the diagnosis-ready app artifacts into a typed object with clear relationships between:

- overall portfolio risk
- top concerns
- supporting metrics
- holding drivers
- sector drivers
- evidence gaps


## Why this notebook exists

The current app already measures risk. This notebook adds the next layer: diagnosis.

In plain English:
- the app's `risk_score` tells us **how much risk** the portfolio appears to have
- `PortfolioRiskDiagnosis` should tell us **what kind of risk it is, what is driving it, and how confident we are in that read**

This notebook is deliberately object-first instead of dataframe-first so we can build reusable relationships before we move into sell logic or rebalancing.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
DIAGNOSIS_DIR = ROOT / 'data' / 'processed' / 'diagnosis'
OUTPUT_PATH = DIAGNOSIS_DIR / 'portfolio_risk_diagnosis_first_stab.json'

from portfolio_analyzer.diagnosis import portfolio_risk_diagnosis_from_saved_artifacts

print('Diagnosis directory:', DIAGNOSIS_DIR)
print('Exists:', DIAGNOSIS_DIR.exists())


In [ ]:
diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)

print('Run ID:', diagnosis.run_id)
print('Observed risk:', diagnosis.observed_risk_score, diagnosis.observed_risk_band)
print('Stated risk:', diagnosis.stated_risk_score, diagnosis.stated_risk_band)
print('Alignment:', diagnosis.alignment)
print('Confidence:', diagnosis.confidence_band)
print()
print('Diagnostic summary:')
print(diagnosis.diagnostic_summary)


In [ ]:
diagnosis.model_dump()


In [ ]:
concerns_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_concerns])
holding_drivers_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_holding_drivers])
sector_drivers_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_sector_drivers])
supporting_metrics_df = pd.DataFrame([item.model_dump() for item in diagnosis.supporting_metrics])

display(concerns_df)
display(holding_drivers_df)
display(sector_drivers_df)
display(supporting_metrics_df[['metric_key', 'group', 'label', 'raw_value', 'score', 'score_readout']])


## First-stab gaps

This object is intentionally incomplete. It already gives us a clean typed diagnosis layer, but the next version should improve on:

- linking filing/news evidence directly into concern ranking
- adding a dedicated macro regime summary object
- modeling user goals and constraints beyond the stated risk score
- refining how holding-level drivers are ranked and explained


In [ ]:
OUTPUT_PATH.write_text(json.dumps(diagnosis.model_dump(), indent=2) + '\n', encoding='utf-8')
print('Saved first-stab diagnosis object to:')
print(OUTPUT_PATH)
